# Tutorial: from data to dashboard

In this tutorial you will find glucose data in the JupyterHealth Exchange, measure how much of the time a patient was in range, plot it, and share it as a dashboard that anyone on the Hub can open without seeing any code.

It takes about 15 minutes.
You need an [Exchange account](sign-up.md) that can [log in to the Hub](log-in.md).

{button}`Open this tutorial on the Hub <https://jupyter-health.2i2c.cloud/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2Fjupyterhealth%2Fjupyterhealth-hub-docs&branch=add%2Ftutorial&urlpath=lab%2Ftree%2Fjupyterhealth-hub-docs%2Fdocs%2Ftutorial.ipynb>`

Run each code cell in order with {kbd}`Shift+Enter`.

## 1. Connect to the Exchange

The Hub has already given your session a token, so no password is needed.

In [ ]:
from jupyterhealth_client import Code, JupyterHealthClient

client = JupyterHealthClient()

## 2. Find a study and a patient

List the studies you can see:

In [ ]:
for study in client.list_studies():
    print(study["id"], study["name"])

We'll use the **Sample Data** study, which has synthetic patients.
List its patients:

In [ ]:
study_id = 30011

for patient in client.list_patients(study_id=study_id):
    print(patient["id"], patient["nameGiven"], patient["nameFamily"])

Pick one patient:

In [ ]:
patient_id = 40072

## 3. Load their glucose readings

In [ ]:
import pandas as pd

df = client.list_observations_df(
    patient_id=patient_id,
    study_id=study_id,
    code=Code.BLOOD_GLUCOSE,
    limit=10_000,
)
glucose = pd.DataFrame(
    {
        "time": pd.to_datetime(df["effective_time_frame_date_time_local"]),
        "glucose": df["blood_glucose_value"],
    }
).sort_values("time")
glucose.head()

Each row is one reading from a continuous glucose monitor, in mg/dL.

## 4. Measure time in range

The target range for glucose is 70–180 mg/dL.
Calculate the average reading and the share of readings in that range:

In [ ]:
mean_glucose = glucose["glucose"].mean()
in_range = glucose["glucose"].between(70, 180).mean() * 100

print(f"Average glucose: {mean_glucose:.0f} mg/dL")
print(f"Time in range: {in_range:.0f}%")

## 5. Plot it

Draw the readings over time, with the target range shaded green:

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
ax.axhspan(70, 180, color="green", alpha=0.15, label="Target range")
ax.plot(glucose["time"], glucose["glucose"], linewidth=0.8)
ax.set_ylabel("Glucose (mg/dL)")
ax.set_title(f"Patient {patient_id}: {in_range:.0f}% of time in range")
ax.legend(loc="upper right");

## 6. Share it as a dashboard

Steps 1–5 are already packaged as a dashboard in [`glucose-dashboard.ipynb`](glucose-dashboard.ipynb), with a dropdown to choose the patient.

1. In the file browser on the left, open `glucose-dashboard.ipynb`.
   It's the same code you just ran.
2. Open this link in a new browser tab:

   ```text
   https://jupyter-health.2i2c.cloud/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2Fjupyterhealth%2Fjupyterhealth-hub-docs&branch=add%2Ftutorial&urlpath=voila%2Frender%2Fjupyterhealth-hub-docs%2Fdocs%2Fglucose-dashboard.ipynb
   ```

   The Hub runs the notebook and shows only its results: no code, no JupyterLab.
3. Send the same link to a colleague, for example:

   > Here's a glucose dashboard for our sample study.
   > Click the link, log in with your JupyterHealth Exchange account, and wait a minute while it loads.

Your colleague needs an Exchange account that can log in to the Hub, but they don't need to know anything about Jupyter or Python.
After they click the link, they see:

1. The Hub login page. They click {guilabel}`Log in to continue`.

   ```{figure} images/hub-login.png
   :alt: The Hub landing page with a Log in to continue button
   :width: 400px
   ```

2. The Exchange login form. They enter their email and password.

   ```{figure} images/exchange-login.png
   :alt: The JupyterHealth Exchange log in form
   :width: 250px
   ```

3. A "server starting" progress bar, then a "pulling files" progress bar.
4. The dashboard.

The link always opens the version of the dashboard in [this repository](https://github.com/jupyterhealth/jupyterhealth-hub-docs).
To share a dashboard of your own, put the notebook in a public GitHub repository and change `repo=` and the path in the link.

## What you did

- Found a study and a patient in the Exchange, and loaded their glucose readings.
- Measured time in range and plotted it.
- Shared the result as a dashboard that a colleague can open with a single link.

Next:

- [Run a simple analysis](run-an-analysis.md) covers the client in more detail.
- The [CGM tutorial](https://jupyterhealth.github.io/software-documentation/tutorial/tutorial-cgm) computes more glucose metrics.
- The [client API reference](https://jupyterhealth-client.readthedocs.io) lists the other data types you can load.